<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2-power-grid-stability-prediction/Ex12.2_00_system_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.2 · Notebook 00 — System Check

**Paired with L12.2 · Prediction of Power Grid Stability**

Run this first, top to bottom. Nothing to write: every cell is complete. Its
job is to establish, before you spend an hour on anything else, that

* `torch`, `numpy` and `matplotlib` import and are recent enough;
* the three modules beside this notebook load — `course_core`, `pinn_core`,
  `problem`;
* the six-bus network here is **the same network** as in Ex_05 notebook 03,
  L5.1 and Ex_12.1 — printed, not asserted;
* the contingency list is what you think it is, islanding cases included;
* and that **one** critical clearing time, computed honestly, costs what it
  costs.

That last number is the point of the whole exercise set. Everything after this
notebook exists because it is too large.

If a cell fails here, fix it before going on.

---

## 0 · Versions

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2-power-grid-stability-prediction/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import sys
import numpy as np
import matplotlib
import torch

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("torch      ", torch.__version__)
print("cuda        available:", torch.cuda.is_available(), " (not needed)")

**What you should see.** Four version numbers. Any Python from 3.9 and any
PyTorch from 2.0 will do, and **no GPU is required anywhere in Part 2**.

This set also needs `scipy` — `problem.equilibrium` uses `scipy.optimize.brentq`
to place the machines at an exact pre-fault equilibrium. Colab has it.

---

## 1 · The three modules

Every Part 2 exercise has the same three files beside it. The first two are
identical in every set; only the third changes.

| | |
|---|---|
| `course_core.py` | shared by the whole course — `set_seed`, `MLP`, `to_tensor`, `check` |
| `pinn_core.py` | the PDE machinery — `grad`, `d2`, samplers, `train_two_stage` |
| `problem.py` | **this** problem — the network, the contingencies, the labels, the graph |

In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

**What you should see.** `device: cpu` on most machines and
`dtype: torch.float64`.

Double precision matters for a different reason here than in Ex_07. There is no
second derivative of a network anywhere in this set. What there is instead is a
**permutation test** in notebook 03, which asks whether two numbers agree to
machine precision. In float32 "machine precision" is about `1e-7`; in float64 it
is about `1e-16`, and a claim of exactness is far more convincing at the second
number than the first.

In [ ]:
import os
os.makedirs(pb.RESULTS, exist_ok=True)
print("results folder:", pb.RESULTS)

---

## 2 · The network, and that it has not changed

The six-bus case appears **four times** in this course:

* **Ex_05 notebook 03** — supervised node regression, injections to state;
* **L5.1** — the worked example on the slides;
* **Ex_12.1** — the object of a physics-informed state estimator;
* **here** — the thing being screened for stability.

If those were four different networks that happened to have six buses, the
repetition would teach nothing. They are one network, carried through the
course, and `check_continuity` prints it so you can see rather than trust.

In [ ]:
buses, branches = pb.check_continuity()

print()
print("  the same list, as Ex_12.1 would print it:")
print("   ", buses[1])
print("   ", branches[0])

**What you should see.** Six buses — slack, one generator, three loads and the
German HVDC link entering as a fixed injection — and six lines with
reactances between 0.030 and 0.046 p.u.

If you have Ex_12.1 to hand, the honest check is a diff:

```
diff <(sed -n '/^BUSES = \[/,/^\]/p' ../Ex12.1-power-grid-stability-estimation/problem.py) \
     <(sed -n '/^BUSES = \[/,/^\]/p' problem.py)
```

and the same for `BRANCHES`. Both should print nothing.

**Provenance, before you quote any of it.** The case is *DK2-representative,
not DK2*. The structure follows eastern Denmark; the line impedances are
plausible textbook values, not measured ones, because no TSO publishes a nodal
model with impedances. Conclusions that depend on the impedances have to say
so. Conclusions that depend only on structure — which contingency is worst, how
many hops a disturbance travels — are on firmer ground.

In [ ]:
pb.plot_network(title="The six-bus network — intact")
plt.show()

pb.plot_network(outage=0, fault_bus=0,
                title="Contingency 1: fault at bus 0, cleared by tripping line 0")
plt.show()

**What you should see.** Two pictures of the same graph, the second with line 0
struck through in orange and bus 0 ringed. Line 0 is the direct tie between the
slack (the Swedish AC link) and Gen east. Losing it is the interesting case and
you will see why in section 6.

---

## 3 · The admittance matrix **is** the graph

There is no separate edge list in this exercise, and that is deliberate: an
edge list and an admittance matrix can drift out of step, and then the picture
and the physics disagree with nobody the wiser. `pb.adjacency` reads the
structure straight off `build_ybus`.

In [ ]:
Y  = pb.build_ybus()
Y0 = pb.build_ybus(outage=0)

print("Y[0,1] intact   :", np.round(Y[0, 1], 4))
print("Y[0,1] line 0 out:", np.round(Y0[0, 1], 4))
print("off-diagonal non-zeros:", int((np.abs(Y) > 1e-12).sum()),
      "->", int((np.abs(Y0) > 1e-12).sum()))
print("symmetric:", np.allclose(Y, Y.T), np.allclose(Y0, Y0.T))
print()

A = pb.adjacency()
print("adjacency, intact:\n", A.astype(int))
print("degree, intact    :", A.sum(axis=1).astype(int))
print("degree, line 0 out:", pb.adjacency(outage=0).sum(axis=1).astype(int))


def hop_distance(A):
    "Shortest path length in hops between every pair of buses."
    n = A.shape[0]
    D = np.full((n, n), np.inf)
    np.fill_diagonal(D, 0)
    reach = np.eye(n, dtype=bool)
    for h in range(1, n + 1):
        reach = (reach @ (A > 0)) > 0
        D[np.logical_and(reach, np.isinf(D))] = h
    return D


D_intact = hop_distance(A)
D_out0 = hop_distance(pb.adjacency(outage=0))
print()
print("hop distances, intact:\n", D_intact.astype(int))
print("diameter, intact    :", int(D_intact.max()))
print("diameter, line 0 out:", int(D_out0.max()))

**What you should see.** `Y[0,1] = -3.8367+32.8857j` intact and exactly `0j`
with line 0 out; eighteen non-zero entries falling to sixteen; both matrices
symmetric.

And two numbers that will decide an architecture in notebook 03:

* **diameter 3** for the intact network,
* **diameter 4** once line 0 is out.

A message-passing layer moves information one hop. So a graph network with
fewer than three layers cannot let bus 5's load reach bus 0's machine at all,
and with fewer than four it cannot do so on the very topology that matters
most. Depth here is a property of the graph, not a hyperparameter to tune
blindly. L5.1 made that argument; this is the network it applies to.

---

## 4 · The operating point

The pre-fault state comes from a Newton-Raphson power flow — Ex_12.1's, copied
across unchanged. Everything downstream depends on it, so check it here.

In [ ]:
P, Q = pb.injections()
V, th, converged, iters = pb.solve_power_flow(Y, P, Q)

print("converged:", converged, " in", iters, "iterations")
print("|V|   :", np.round(V, 5))
print("angle :", np.round(np.degrees(th), 4), "deg")

P_calc, Q_calc = pb.pq_from_state(V, th, Y)
print()
print(f"slack injection : {P_calc[0]:+.5f} p.u.  = {P_calc[0]*pb.BASE_MVA:+.2f} MW")
print(f"network losses  : {P_calc.sum():+.5f} p.u.  = {P_calc.sum()*pb.BASE_MVA:+.2f} MW")
print(f"total load      : {-P[[2,3,4]].sum():.3f} p.u.")

mismatch = np.max(np.abs(np.concatenate([P[1:] - P_calc[1:], Q[1:] - Q_calc[1:]])))
check("power flow mismatch at buses 1-5", mismatch, 0.0, tol=1e-10)

**What you should see.** Convergence in **5 iterations**; voltages between
0.9706 and 1.000 p.u.; angles inside ±2.5°; a slack injection of **+0.59505
p.u. = +59.51 MW** imported through the Swedish link, and losses of
**+0.01505 p.u. = 1.51 MW**, which is 0.7 % of the 2.09 p.u. load. Both are
the right order for a compact 400 kV network.

The mismatch check should PASS at `1e-10`; the value printed is around `2e-14`,
which is the Newton solve converged to double precision rather than to its
tolerance.

---

## 5 · The contingency set

An **N-1 contingency** here is the textbook one:

> a solid three-phase fault at the sending end of branch *k*,
> cleared at time $t_c$ by tripping branch *k*.

The fault location moves with the contingency because it has to: you do not
clear a fault on one line by tripping a different one.

Not every branch qualifies. Removing some of them does not stress the system —
it disconnects part of it, and there is no stability question left to ask.
`contingencies()` finds those from the connectivity of the admittance matrix
and drops them.

In [ ]:
cases = pb.contingencies(verbose=True)

print()
for c in cases:
    print(f"   {c['index']}  outage={str(c['outage']):>4s}  "
          f"fault at bus {c['fault_bus']}   {c['label']}")

print()
print("islanding outages:", pb.islanding_outages())
print("N_CONTINGENCY    :", pb.N_CONTINGENCY)

**What you should see.** Six branches, **one skipped as islanding** — branch 5,
the radial spur from Load city to the German HVDC link — leaving
**six contingencies**: the base case plus five outages.

Branch 5 is skipped because bus 5 hangs off it with no other connection. Remove
it and bus 5 has no path to any generator. That is a **loss-of-supply**
contingency, not a stability contingency: the customers on bus 5 are already in
the dark before any machine has had time to swing, and a screening tool that
scored it on rotor-angle stability would be answering the wrong question about
the right event. Real screening reports the two categories separately.

The base case is the same fault at Gen east's terminals cleared *without*
losing a line — a successful autoreclose. It is the reference against which
the cost of actually losing a branch is read.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14.5, 7.0))
for c, ax in zip(cases, axes.ravel()):
    pb.plot_network(outage=c["outage"], fault_bus=c["fault_bus"], ax=ax,
                    title=f"{c['index']} · {c['label']}")
fig.tight_layout()
plt.show()

---

## 6 · One critical clearing time, computed honestly

This is the ground truth of the entire exercise set. For one operating point
and one contingency:

1. solve the pre-fault power flow;
2. compute each machine's internal EMF behind its transient reactance;
3. Kron-reduce three networks onto the machine internal nodes — pre-fault,
   fault-on, post-fault;
4. place the machines at an exact equilibrium;
5. **bisect on the clearing time**, integrating the swing equations by RK4 at
   every trial, until the longest survivable fault is bracketed to 2 ms.

Step 5 is the expensive one: ten or eleven full simulations per label.

In [ ]:
import time

op = np.stack(pb.injections())          # the base dispatch, as an operating point
c = cases[1]                            # trip line 0 — the tie to Sweden

t0 = time.time()
cct, detail = pb.label_case(op, c["outage"], c["fault_bus"], return_detail=True)
elapsed = time.time() - t0

print(c["label"])
print(f"  critical clearing time : {cct:.4f} s   =  {cct*1e3:.1f} ms"
      f"  =  {cct*50:.1f} cycles at 50 Hz")
print(f"  protection can deliver : {pb.PROTECTION_TIME*1e3:.0f} ms")
print(f"  secure                 : {bool(pb.secure(cct))}")
print(f"  wall-clock cost        : {elapsed:.3f} s   for ONE contingency")
print()
print("  internal EMF magnitudes:", np.round(detail["machines"].E, 5))
print("  pre-fault rotor angles :", np.round(np.degrees(detail["delta0"]), 4), "deg")
print("  mechanical power       :", np.round(detail["Pm"], 5), "p.u.")
print()
print(f"  |Y_transfer| pre-fault : {abs(detail['Yred_pre'][0,1]):.4f}")
print(f"  |Y_transfer| fault-on  : {abs(detail['Yred_fault'][0,1]):.6f}")
print(f"  |Y_transfer| post-fault: {abs(detail['Yred_post'][0,1]):.4f}")

**What you should see.** A critical clearing time of **0.2197 s = 219.7 ms**,
about **11 cycles** at 50 Hz — comfortably longer than the 140 ms the
protection can deliver, so this contingency is **secure at the base dispatch**.
Note that word: *at the base dispatch*. Notebook 01 will load the system up and
the answer will change.

The three transfer admittances tell the story of the event in three numbers:
**2.835** before the fault, **0.0152** during it — the fault has all but
disconnected the machine from the system, so it accelerates on its mechanical
power alone — and **1.977** afterwards, weaker than before because the tie to
Sweden is now gone. The machine must give back, into a weaker network, the
energy it picked up during the fault. Whether it can is the question.

And the cost: about **0.4 s of CPU for one number**. Hold on to that.

In [ ]:
# Verify that the pre-fault state really is an equilibrium: at delta0, the
# electrical power out of each machine must equal its mechanical power in.
Pe0 = pb.electrical_power(detail["delta0"], detail["machines"].E,
                          detail["Yred_pre"])
check("machine 1 is in equilibrium before the fault", Pe0[1], detail["Pm"][1],
      tol=1e-9)

# And the equilibrium angle is not a coincidence: it is the argument of the
# internal EMF, which the power flow already knew.
E = pb.internal_emf(detail["V"], detail["th"], detail["P_calc"],
                    detail["Q_calc"], detail["machines"])
delta_from_emf = np.angle(E[1]) - np.angle(E[0])
check("equilibrium angle = arg(E1) - arg(E0)", detail["delta0"][1],
      delta_from_emf, tol=1e-9)
print(f"  both give {np.degrees(delta_from_emf):.4f} deg")

**What you should see.** Two PASSes, and **18.6830 deg** printed twice.

That agreement is worth a moment. The rotor angle was obtained two completely
different ways — once by a root find on the reduced network's power equation,
once by taking the argument of a complex number the power flow handed over —
and they agree to nine decimal places. The classical machine model and the
power flow are the same model, seen from two ends. If you ever change `X_D_PRIME`
or the load conversion and this check starts failing, the two ends have come
apart and every CCT after it is wrong.

---

### The swing curves either side of the answer

A critical clearing time is a boundary. The most useful thing you can do with
one is to step across it.

In [ ]:
for offset, tag in [(-0.010, "10 ms below the CCT"), (+0.010, "10 ms above the CCT")]:
    tc = cct + offset
    T, D, W = pb.simulate_swing(detail["machines"], detail["Yred_pre"],
                                detail["Yred_post"],
                                t_end=pb.T_END, t_fault=pb.T_FAULT,
                                t_clear=pb.T_FAULT + tc,
                                Yred_fault=detail["Yred_fault"],
                                delta0=detail["delta0"])
    sep = np.degrees(D[:, 1] - D[:, 0])
    pb.plot_swing(T, D, W, t_fault=pb.T_FAULT, t_clear=pb.T_FAULT + tc,
                  title=f"clearing at {tc*1e3:.1f} ms — {tag}")
    plt.show()
    print(f"  {tag:22s}  max separation {sep.max():8.1f} deg"
          f"   at t = {T[int(np.argmax(sep))]:.3f} s"
          f"   final {sep[-1]:8.1f} deg")

**What you should see.** Two figures that could not be more different.

* Cleared at **209.7 ms**, the rotor swings out to **122.0°** — past the 90°
  that a static analysis would call the limit, which is exactly why transient
  stability is not a static question — turns round, and oscillates back. The
  machine stayed in step.
* Cleared at **229.7 ms**, twenty milliseconds later, the angle runs away past
  **553.9°** and keeps going. The machine has slipped a pole. In reality its
  protection would trip it off the system, and the load it was carrying would
  have to come from somewhere in the next few seconds.

Twenty milliseconds — **one cycle** — is the whole difference between those two
pictures. That is what "critical" means, and it is why the number is worth
computing carefully.

Look at the frequency panel too: in the stable case machine 1 swings between
roughly 48.4 and 51.4 Hz and comes back; in the unstable case it climbs and
does not.

---

## 7 · What the honest answer costs

You have now computed one label. The exercise needs rather more than one.

In [ ]:
t0 = time.time()
base_cct = [pb.label_case(op, c["outage"], c["fault_bus"]) for c in cases]
sweep = time.time() - t0

print("the full contingency list at the base dispatch:\n")
for c, v in zip(cases, base_cct):
    flag = "secure" if pb.secure(v) else "INSECURE"
    ceil = "  (censored at CCT_MAX)" if v >= pb.CCT_MAX else ""
    print(f"  {c['label']:<46s} {v*1e3:6.1f} ms   {flag}{ceil}")

per_label = sweep / len(cases)
print(f"\n  one screen of six contingencies: {sweep:.2f} s"
      f"   ({per_label*1e3:.0f} ms per label)")
print(f"  a dataset of 180 dispatches    : {sweep*180/60:.1f} min (notebook 01 does this)")
print()
n_branch, n_disp = 4000, 24
print(f"  a real TSO: {n_branch} branches screened {n_disp} times a day")
print(f"     {n_branch*n_disp:,} labels")
for factor, what in [(1, "at this six-bus cost"),
                     (100, "at 100x the cost — a few hundred buses"),
                     (10000, "at 10000x — a full synchronous area")]:
    print(f"       {n_branch*n_disp*per_label*factor/3600:12,.0f} CPU-hours per day"
          f"   {what}")

**What you should see.** At the base dispatch every contingency is secure, with
clearing times of **235.4, 219.7, 235.4, 500.0, 245.1 and 500.0 ms**, and a
sweep that takes a second or two — about 270 ms per label on the machine this
was written on. Two of
them hit the ceiling: `CCT_MAX = 500 ms` is where the bisection stops, because a
contingency that survives half a second of solid three-phase fault is not a
stability constraint — the breakers would have cleared it four times over. Those
two are **censored**, and notebook 01 says what that does to a regression target.

Three things follow from the timings.

**One screen is about two seconds.** Fine — do it once.

**A training set is minutes.** Notebook 01 spends about five and a half of them,
once, and caches the result.

**A real control room cannot do this at all.** Four thousand branches screened
every hour is about **96,000 labels a day**. At this six-bus model's cost that
is only seven CPU-hours, which sounds survivable — and that number is the trap.
The cost per label is not a constant. This model has two machines and six buses
and integrates for one second; a dynamic model of a synchronous area has
hundreds of machines and tens of thousands of buses, its right-hand side is
thousands of times more expensive, and it is integrated for ten seconds rather
than one. The last row of the printout is the realistic one, and it does not
finish before the operating point moves.

The industry's actual answer is a mixture of screening filters, DC
approximations and pre-computed nomograms — and, increasingly, the surrogate you
are about to build.

**What a surrogate is allowed to replace.** Not the simulator. The simulator
stays; it is what generated these labels and it is what you go back to for the
contingencies the surrogate flags. What the surrogate replaces is the
*exhaustive sweep*: it looks at all four thousand and hands you the twenty worth
simulating. Notebook 04 is about getting that hand-off right, and about what it
costs when you get it wrong.

---

## 8 · Before you move on

1. `contingencies()` skipped one branch. Say in one sentence why a
   loss-of-supply contingency and a transient-stability contingency should not
   be scored on the same axis.
2. The transfer admittance fell from 2.835 to 0.0152 during the fault and came
   back to 1.977 rather than 2.835. Which of those two changes sets the CCT,
   and why is a fault at a *machine bus* the severe case?
3. The intact network has diameter 3 and loses a hop when line 0 is out. Give
   the minimum number of message-passing layers a graph model needs here, and
   say what a model with two would be structurally unable to represent.
4. Two of the six base-case labels were censored at 500 ms. Name one thing that
   would go wrong if you trained a regression on the censored values as if they
   were measurements.

Next: **notebook 01**, where the dataset gets built and the base case stops
looking so comfortable.